# Raw Ingestion — Food Inspections (dev)

Lands `food_inspections_dev.csv` from the `raw` volume into the `bronze` volume as Parquet, **as-is**: every column is read as STRING (no type conversion, no cleaning, no filtering). Files are organized as `bronze/food_inspections/<ingest-date>/`.

In [0]:
# Source and destination paths (bronze layer: raw, untyped landing zone)
from datetime import date

SOURCE_CSV = "/Volumes/workspace/foodguard_ai/raw/food_inspections_dev.csv"
BRONZE_BASE = "/Volumes/workspace/foodguard_ai/bronze/food_inspections"
INGEST_DATE = date.today().isoformat()
TARGET_PATH = f"{BRONZE_BASE}/{INGEST_DATE}"

print(f"Source : {SOURCE_CSV}")
print(f"Target : {TARGET_PATH}")

In [0]:
# Raw read: no schema, no inference — every column stays exactly as in the CSV (STRING).
# multiLine=True keeps records intact: some quoted fields (violations, location) contain embedded newlines.
raw_df = (
    spark.read
    .option("header", True)
    .option("multiLine", True)
    .csv(SOURCE_CSV)
)

raw_df.printSchema()
print(f"Rows read: {raw_df.count()}")
display(raw_df.limit(5))

In [0]:
# Write as Parquet, unmodified: single part file per daily folder, overwrite on re-run.
(
    raw_df.coalesce(1)
    .write
    .mode("overwrite")
    .parquet(TARGET_PATH)
)

print(f"Ingested to {TARGET_PATH}")

In [0]:
# Sanity check: re-read the Parquet and confirm row count and all-STRING schema survived.
check_df = spark.read.parquet(TARGET_PATH)

src_count = raw_df.count()
out_count = check_df.count()
same_schema = raw_df.schema == check_df.schema

print(f"Rows  — source: {src_count} | parquet: {out_count} | match: {src_count == out_count}")
print(f"Schema match (all STRING): {same_schema}")
assert src_count == out_count, "Row count mismatch!"
assert same_schema, "Schema mismatch — types were altered!"
display(check_df.limit(5))